<a href="https://colab.research.google.com/github/AKSHAYA404/NLP/blob/main/2403A54118_LAB_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import numpy as np

In [20]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [21]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [22]:
from sklearn.decomposition import LatentDirichletAllocation, NMF

In [23]:
import pprint

In [24]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
import pandas as pd
data= pd.read_csv('/content/sample_nlp.csv')
data.head()

,news
0,Virat scored century in match
1,BJP won in election
2,Bumra took five wickets in a match
3,Congress form state government


In [8]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocessing_pipeline(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "\U0001f926-\U0001f937"
        "\U00010000-\U0010ffff"
        "\u2640-\u2642"
        "\u2600-\u2B55"
        "\u200d"
        "\u23cf"
        "\u23e9"
        "\u231a"
        "\ufe0f"
        "\u3030"
        "]+", re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    preprocessed_text = ' '.join(lemmatized_tokens)
    return preprocessed_text
data['preprocessed_text'] = data['news'].apply(preprocessing_pipeline)
data.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,news,preprocessed_text
0,Virat scored century in match,virat scored century match
1,BJP won in election,bjp election
2,Bumra took five wickets in a match,bumra took five wicket match
3,Congress form state government,congress form state government


In [9]:
def bag_of_words(corpus):
    from sklearn.feature_extraction.text import CountVectorizer
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(corpus)
    try:
        feature_names = vectorizer.get_feature_names_out()
    except AttributeError:
        feature_names = vectorizer.get_feature_names()
    bow_df = pd.DataFrame(X.toarray(), columns=feature_names)
    return bow_df

In [10]:
bag_of_words_df = bag_of_words(data['preprocessed_text'])
print(bag_of_words_df.head(10))

   bjp  bumra  century  congress  election  five  form  government  match  \
0    0      0        1         0         0     0     0           0      1   
1    1      0        0         0         1     0     0           0      0   
2    0      1        0         0         0     1     0           0      1   
3    0      0        0         1         0     0     1           1      0   

   scored  state  took  virat  wicket  
0       1      0     0      1       0  
1       0      0     0      0       0  
2       0      0     1      0       1  
3       0      1     0      0       0  


In [11]:
from sklearn.decomposition import LatentDirichletAllocation
num_topics = 2
LDA=LatentDirichletAllocation(n_components=num_topics, random_state=42)
LDA.fit(bag_of_words_df)

LatentDirichletAllocation(n_components=2, random_state=42)

In [12]:
def display_topics(model, feature_names, num_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic {topic_idx}:")
        top_features_ind = topic.argsort()[:-num_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        print(" ".join(top_features))
    print("\n")


In [13]:
num_top_words = 15
print(f"Top {num_top_words} words for each topic:")
display_topics(LDA, bag_of_words_df.columns, num_top_words)

Top 15 words for each topic:
Topic 0:
match five took wicket bumra virat century scored election bjp state form government congress
Topic 1:
congress government state form bjp election scored century virat match bumra wicket took five




In [14]:
data['news']= data['news'].astype(str)
data['LDA_Topic'] = LDA.transform(bag_of_words_df).argmax(axis=1)
print(data[['news', 'LDA_Topic']].head())

                                 news  LDA_Topic
0       Virat scored century in match          0
1                 BJP won in election          0
2  Bumra took five wickets in a match          0
3      Congress form state government          1


In [15]:
from sklearn.decomposition import NMF
num_topics = 2
nmf_model = NMF(n_components=num_topics, random_state=42)
nmf_model.fit(bag_of_words_df)

NMF(n_components=2, random_state=42)

In [16]:
print(f"Top {num_top_words} words for each topic using NMF:")
display_topics(nmf_model, bag_of_words_df.columns, num_top_words)

Top 15 words for each topic using NMF:
Topic 0:
match wicket took five bumra virat century scored state government election form congress bjp
Topic 1:
state government form congress election bjp virat century scored match took wicket five bumra




In [17]:
data['news']= data['news'].astype(str)
document_topics = nmf_model.transform(bag_of_words_df)
data['NMF_Topic'] = document_topics.argmax(axis=1)
print(data[['news', 'NMF_Topic']].head())

                                 news  NMF_Topic
0       Virat scored century in match          0
1                 BJP won in election          1
2  Bumra took five wickets in a match          0
3      Congress form state government          1


In [18]:
data.head()

,news,preprocessed_text,LDA_Topic,NMF_Topic
0,Virat scored century in match,virat scored century match,0,0
1,BJP won in election,bjp election,0,1
2,Bumra took five wickets in a match,bumra took five wicket match,0,0
3,Congress form state government,congress form state government,1,1


In [34]:
df = pd.read_csv("/content/arxiv_data.csv", on_bad_lines='warn', engine='python')
print(df.head())
print(df.columns)

                                              titles  \
0  Survey on Semantic Stereo Matching / Semantic ...   
1  FUTURE-AI: Guiding Principles and Consensus Re...   
2  Enforcing Mutual Consistency of Hard Regions f...   
3  Parameter Decoupling Strategy for Semi-supervi...   
4  Background-Foreground Segmentation for Interio...   

                                           summaries  \
0  Stereo matching is one of the widely used tech...   
1  The recent advancements in artificial intellig...   
2  In this paper, we proposed a novel mutual cons...   
3  Consistency training has proven to be an advan...   
4  To ensure safety in automated driving, the cor...   

                         terms  
0           ['cs.CV', 'cs.LG']  
1  ['cs.CV', 'cs.AI', 'cs.LG']  
2           ['cs.CV', 'cs.AI']  
3                    ['cs.CV']  
4           ['cs.CV', 'cs.LG']  
Index(['titles', 'summaries', 'terms'], dtype='object')


/tmp/ipython-input-2651259073.py:1: ParserWarning: Skipping line 5600: unexpected end of data

  df = pd.read_csv("/content/arxiv_data.csv", on_bad_lines='warn', engine='python')


In [35]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www.\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return " ".join(tokens)
df["clean_abstract"] = df["summaries"].apply(lambda x: clean_text(str(x)))
print(df["clean_abstract"].head())

0    stereo matching one widely used technique infe...
1    recent advancement artificial intelligence com...
2    paper proposed novel mutual consistency networ...
3    consistency training proven advanced semisuper...
4    ensure safety automated driving correct percep...
Name: clean_abstract, dtype: object


In [36]:
vectorizer = CountVectorizer(
    max_df=0.95,
    min_df=2,
    stop_words='english'
)
bow = vectorizer.fit_transform(df["clean_abstract"])

In [37]:
bow_df = pd.DataFrame(
    bow.toarray(),
    columns = vectorizer.get_feature_names_out()
)
print(bow_df.shape)
print(bow_df.head())

(5598, 9978)
   aae  aaf  abbreviate  abbreviated  abdomen  abdominal  aberration  ability  \
0    0    0           0            0        0          0           0        0   
1    0    0           0            0        0          0           0        0   
2    0    0           0            0        0          0           0        0   
3    0    0           0            0        0          0           0        1   
4    0    0           0            0        0          0           0        0   

   ablate  ablation  ...  zernike  zero  zeroshot  zone  zoo  zoom  zooming  \
0       0         0  ...        0     0         0     0    0     0        0   
1       0         0  ...        0     0         0     0    0     0        0   
2       0         0  ...        0     0         0     0    0     0        0   
3       0         0  ...        0     0         0     0    0     0        0   
4       0         0  ...        0     0         0     0    0     0        0   

   zsi  zsl  zurich  
0  

In [38]:
num_topics = 10

lda_model = LatentDirichletAllocation(
    n_components=num_topics,
    random_state=42,
    learning_method='batch'
)

In [39]:
def display_topics(model, feature_names, no_top_words):
    for idx, topic in enumerate(model.components_):
        print(f"Topic {idx+1}: ")
        print([ feature_names[i] for i in topic.argsort()[:-no_top_words-1:-1] ])

lda_model.fit(bow)
display_topics(lda_model, vectorizer.get_feature_names_out(), 10)


Topic 1: 
['image', 'segmentation', 'method', 'algorithm', 'problem', 'based', 'clustering', 'proposed', 'approach', 'result']
Topic 2: 
['learning', 'representation', 'data', 'task', 'method', 'domain', 'model', 'training', 'feature', 'selfsupervised']
Topic 3: 
['image', 'model', 'text', 'segmentation', 'language', 'dataset', 'visual', 'quality', 'description', 'word']
Topic 4: 
['graph', 'representation', 'node', 'learning', 'network', 'method', 'task', 'model', 'structure', 'information']
Topic 5: 
['image', 'video', 'segmentation', 'method', 'human', 'network', 'frame', 'feature', 'motion', 'result']
Topic 6: 
['image', 'network', 'method', 'model', 'generation', 'training', 'data', 'adversarial', 'object', 'generative']
Topic 7: 
['detection', 'model', 'data', 'attack', 'network', 'adversarial', 'time', 'approach', 'temporal', 'anomaly']
Topic 8: 
['object', 'detection', 'feature', 'method', 'model', 'information', 'propose', 'proposed', 'performance', 'network']
Topic 9: 
['segm

In [40]:
topic_assignments = lda_model.transform(bow)
df["lda_topic"] = topic_assignments.argmax(axis=1)

df[["clean_abstract", "lda_topic"]].head()

,clean_abstract,lda_topic
0,stereo matching one widely used technique infe...,7
1,recent advancement artificial intelligence com...,8
2,paper proposed novel mutual consistency networ...,1
3,consistency training proven advanced semisuper...,1
4,ensure safety automated driving correct percep...,5


In [41]:
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=2,
    stop_words='english'
)

tfidf = tfidf_vectorizer.fit_transform(df["clean_abstract"])

nmf_model = NMF(
    n_components=num_topics,
    random_state=42
)
nmf_output = nmf_model.fit_transform(tfidf)


In [42]:
display_topics(nmf_model, tfidf_vectorizer.get_feature_names_out(), 10)


Topic 1: 
['learning', 'representation', 'data', 'task', 'contrastive', 'selfsupervised', 'unsupervised', 'method', 'supervised', 'downstream']
Topic 2: 
['graph', 'node', 'representation', 'gnns', 'embedding', 'edge', 'link', 'structure', 'learning', 'embeddings']
Topic 3: 
['segmentation', 'image', 'medical', 'method', 'algorithm', 'semantic', 'region', 'annotation', 'proposed', 'based']
Topic 4: 
['object', 'detection', 'detector', 'scene', 'point', 'box', 'depth', 'method', 'cloud', 'bounding']
Topic 5: 
['image', 'generation', 'generative', 'adversarial', 'gans', 'generator', 'gan', 'generated', 'generate', 'face']
Topic 6: 
['domain', 'adaptation', 'target', 'source', 'data', 'shift', 'transfer', 'alignment', 'crossdomain', 'method']
Topic 7: 
['video', 'temporal', 'action', 'frame', 'motion', 'recognition', 'clip', 'spatiotemporal', 'ucf', 'sequence']
Topic 8: 
['feature', 'attention', 'transformer', 'module', 'information', 'local', 'person', 'global', 'semantic', 'fusion']
Top

In [43]:
df["nmf_topic"] = nmf_output.argmax(axis=1)
df[["clean_abstract", "nmf_topic"]].head()

,clean_abstract,nmf_topic
0,stereo matching one widely used technique infe...,8
1,recent advancement artificial intelligence com...,2
2,paper proposed novel mutual consistency networ...,2
3,consistency training proven advanced semisuper...,7
4,ensure safety automated driving correct percep...,2
